# Fine-tune Mistral-7B with QLoRA for Review Summarization

**Requirements:** Google Colab with T4 GPU (free tier). Optimized to fit in 16GB.

**Upload:** `summary_training_pairs.jsonl` from `data/processed/`

In [ ]:
# Cell 1: Install deps and upload data
!pip install -q transformers datasets peft bitsandbytes accelerate trl

from google.colab import files
uploaded = files.upload()  # Upload summary_training_pairs.jsonl

In [ ]:
# Cell 2: Load and format data
import json
import torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

print(f"GPU: {torch.cuda.get_device_name(0)}")

pairs = []
with open('summary_training_pairs.jsonl') as f:
    for line in f:
        pairs.append(json.loads(line))

print(f"Loaded {len(pairs)} training pairs")

def format_prompt(pair):
    return (
        f"<s>[INST] You are an expert product analyst. "
        f"Given the following customer reviews, write a structured executive summary.\n\n"
        f"{pair['input'][:2000]}\n[/INST]\n"
        f"{pair['summary']}</s>"
    )

formatted = [{'text': format_prompt(p)} for p in pairs]
dataset = Dataset.from_list(formatted)
split = dataset.train_test_split(test_size=0.1, seed=42)
print(f"Train: {len(split['train'])}, Val: {len(split['test'])}")

In [ ]:
# Cell 3: Load Mistral-7B in 4-bit
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print(f"Model loaded: {MODEL_NAME}")

In [ ]:
# Cell 4: Apply LoRA (memory-optimized)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Cell 5: Train (memory-optimized for T4 16GB)
training_args = TrainingArguments(
    output_dir="./mistral-qlora-reviews",
    num_train_epochs=2,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_steps=10,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    bf16=True,
    report_to="none",
    optim="paged_adamw_8bit",
    max_grad_norm=0.3,
    lr_scheduler_type="cosine",
    gradient_checkpointing=True,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=split['train'],
    eval_dataset=split['test'],
    processing_class=tokenizer,
    args=training_args,
)

print("Starting training...")
trainer.train()
print("Training complete!")

In [ ]:
# Cell 6: Save adapter and download
ADAPTER_DIR = "./mistral-qlora-adapter"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

import os
total_size = sum(
    os.path.getsize(os.path.join(ADAPTER_DIR, f))
    for f in os.listdir(ADAPTER_DIR)
) / 1e6
print(f"Adapter size: {total_size:.1f} MB")

!zip -r mistral-qlora-adapter.zip mistral-qlora-adapter/
files.download('mistral-qlora-adapter.zip')

In [ ]:
# Cell 7: Quick test
test_input = pairs[-1]['input'][:2000]

prompt = (
    f"<s>[INST] You are an expert product analyst. "
    f"Given the following customer reviews, write a structured executive summary.\n\n"
    f"{test_input}\n[/INST]\n"
)

inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=300,
        temperature=0.3,
        do_sample=True,
        top_p=0.9,
    )

generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
response = generated.split('[/INST]')[-1].strip()
print("=== Fine-tuned Output ===")
print(response)

In [ ]:
# Cell 8: Training history
history = trainer.state.log_history
for entry in history:
    if 'loss' in entry:
        print(f"Step {entry.get('step', '?')}: loss={entry['loss']:.4f}")
    if 'eval_loss' in entry:
        print(f"  eval_loss={entry['eval_loss']:.4f}")